# Invoice Region Detection and Business Parameter Extraction Using CNN, SSD, IoU, OCR, and Streamlit

## Project overview

This project builds an invoice image-processing pipeline that:

1. Preprocesses invoice images (grayscale, resize, denoise, deskew).
2. Detects invoice visual elements and business-critical regions using CNN / SSD-style
   object detection (bounding boxes), evaluated with IoU.
3. Separately detects **stamp** and **signature** (kept as two distinct labels).
4. Runs OCR **only** on detected regions (not the whole page).
5. Checks for required reference parameters (PO number, order number, contract number,
   project reference, insurance policy number, bill of lading number).
6. Extracts payment terms, due dates, and terms & conditions signals.
7. Outputs a structured JSON record per invoice for a Pistac.io-style obligation-readiness
   workflow, demoed via Streamlit.

## Team & notebooks

| # | Member | Role | Notebook |
|---|---|---|---|
| 1 | Rolando | Data Ingestion, Dataset Management, Data Preparation | `01_rolando_data_ingestion_preparation.ipynb` |
| 2 | Diana | Annotation, Stamp Detection, Signature Detection | `02_diana_stamp_signature_detection.ipynb` |
| 3 | Jordan | Invoice Region Detection, SSD/CNN, IoU Evaluation | `03_jordan_region_detection_iou.ipynb` |
| 4 | Damir | OCR, Parameter Extraction, Terms & Conditions | `04_damir_ocr_parameter_terms_extraction.ipynb` |
| 5 | Hessam | PM, Solution Architect, Integration, Streamlit | `05_hessam_integration_streamlit_demo.ipynb` |

Run them in this order — each stage consumes the previous member's outputs
(see `../model_interface_contract.md` for exact file/schema contracts, and `../runbook.md`
for the full run sequence).

## Pipeline diagram

```
raw invoices --> [Rolando: preprocess] --> manifest.csv
                                              |
                    +-------------------------+-------------------------+
                    v                                                   v
        [Diana: stamp/signature detection]                 [Jordan: region detection + IoU]
                    |                                                   |
                    +---------------------+----------------------------+
                                          v
                          [Damir: OCR on detected crops + parameter/terms extraction]
                                          v
                    [Hessam: integrate -> final JSON -> Streamlit demo]
```


## Quick environment check

In [ ]:
# --- Dataset path setup cell ---
# All paths go through src.config.PATHS (pathlib-based, no hardcoded absolute paths).

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PATHS, load_label_schema, load_required_fields

print("Repo root:", PATHS.repo_root)
print("Raw data dir:", PATHS.raw_dir)
print("Outputs dir:", PATHS.outputs_dir)

# If raw data isn't present yet, download it (see dataset_sources.md for kaggle.json setup):
#   python scripts/download_datasets.py --dataset all


In [ ]:
# Confirm config files load correctly
schema = load_label_schema()
fields = load_required_fields()

print("Region labels:", schema["region_labels"])
print("Visual element labels:", schema["visual_element_labels"])
print("Default required fields:", [f["field_name"] for f in fields["default_required_fields"]])
